In [ ]:
import re
import pandas as pd
import yfinance as yf

In [ ]:
# ==============================================================================
# 1. INPUT TANGGAL EVALUASI & RAW TEXT TRADING PLAN
# ==============================================================================
tanggal_evaluasi = "2026-09-14"  # Tanggal Target Eksekusi / Evaluasi

In [ ]:
# TEMPEL SELURUH TEKS TRADING PLAN DI SINI
raw_text = """

==================================================
 TRADING PLAN: DAAZ.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-11 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-14 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp1,670
. Cut Loss (-3%)    : Rp1,620
. Trailing Trigger  : Rp1,787 (+7%)
. Trailing Lock     : Rp1,754 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: SEMA.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-11 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-14 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp146
. Cut Loss (-3%)    : Rp142
. Trailing Trigger  : Rp156 (+7%)
. Trailing Lock     : Rp153 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: ELIT.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-11 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-14 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp290
. Cut Loss (-3%)    : Rp281
. Trailing Trigger  : Rp310 (+7%)
. Trailing Lock     : Rp304 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: MUTU.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-11 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-14 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp135
. Cut Loss (-3%)    : Rp131
. Trailing Trigger  : Rp144 (+7%)
. Trailing Lock     : Rp142 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: MPIX.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-11 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-14 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp127
. Cut Loss (-3%)    : Rp123
. Trailing Trigger  : Rp136 (+7%)
. Trailing Lock     : Rp133 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: NEST.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-11 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-14 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp750
. Cut Loss (-3%)    : Rp728
. Trailing Trigger  : Rp802 (+7%)
. Trailing Lock     : Rp788 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================

==================================================
 TRADING PLAN: AKSI.JK
==================================================
. Tanggal Sinyal (YYYY-MM-DD): 2026-09-11 (Jumat)
. Target Eksekusi (YYYY-MM-DD): 2026-09-14 (Senin)
--------------------------------------------------
. Harga Beli Acuan : Rp340
. Cut Loss (-3%)    : Rp330
. Trailing Trigger  : Rp364 (+7%)
. Trailing Lock     : Rp357 (+5%)
. Batas Hold Max    : 5 Hari Bursa
==================================================



"""

In [ ]:
# ==============================================================================
# 2. PARSER TRADING PLAN (ANTI-GAGAL SPASI)
# ==============================================================================
def parse_plans(text):
    cleaned_text = text.replace('\xa0', ' ')
    blocks = cleaned_text.split("TRADING PLAN:")
    plans = []

    for block in blocks[1:]:
        ticker = re.search(r"^\s*([A-Za-z0-9\.]+)", block)
        buy = re.search(r"Harga Beli Acuan\s*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)
        cut_loss = re.search(r"Cut Loss[^\n:]*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)
        lock = re.search(r"Trailing Lock[^\n:]*:?\s*Rp\s*([\d,.]+)", block, re.IGNORECASE)

        if ticker and buy and cut_loss and lock:
            code_raw = ticker.group(1).strip()
            ticker_formatted = code_raw if code_raw.endswith(".JK") else f"{code_raw}.JK"
            plans.append({
                "ticker": ticker_formatted,
                "code": ticker_formatted.replace(".JK", ""),
                "buy": int(buy.group(1).replace(",", "").replace(".", "")),
                "cut_loss": int(cut_loss.group(1).replace(",", "").replace(".", "")),
                "lock": int(lock.group(1).replace(",", "").replace(".", ""))
            })
    return plans

parsed_data = parse_plans(raw_text)

if not parsed_data:
    print("X Teks gagal dibaca. Pastikan ada tulisan 'TRADING PLAN: KODE.JK'.")
else:
    df = pd.DataFrame(parsed_data)
    tickers = df["ticker"].tolist()
    print(f"Membaca {len(tickers)} saham... Mengambil data High & Low dari Yahoo Finance...")

    # ==============================================================================
    # 3. AMBIL DATA HIGH DAN LOW PADA TANGGAL EVALUASI
    # ==============================================================================
    t_eval = pd.to_datetime(tanggal_evaluasi)
    start_eval = t_eval.strftime('%Y-%m-%d')
    end_eval = (t_eval + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    downloaded_data = yf.download(tickers, start=start_eval, end=end_eval, group_by='ticker', auto_adjust=False)

    high_dict = {}
    low_dict = {}

    for ticker in tickers:
        try:
            df_tk = downloaded_data if len(tickers) == 1 else downloaded_data[ticker].dropna()
            if not df_tk.empty:
                high_dict[ticker] = df_tk['High'].iloc[-1]
                low_dict[ticker] = df_tk['Low'].iloc[-1]
            else:
                high_dict[ticker] = None
                low_dict[ticker] = None
        except Exception:
            high_dict[ticker] = None
            low_dict[ticker] = None

    df["Harga High"] = df["ticker"].map(high_dict)
    df["Harga Low"] = df["ticker"].map(low_dict)

    # ==============================================================================
    # 4. LOGIKA EVALUASI: UTAMAKAN TERTINGGI (HIGH) BARU TERENDAH (LOW)
    # ==============================================================================
    def evaluasi_prioritas(row):
        high = row["Harga High"]
        low = row["Harga Low"]

        if pd.isna(high) or pd.isna(low):
            return "DATA TIDAK ADA"
        # Prioritas 1: Cek apakah pernah menyentuh/melewati target Profit
        if high >= row["lock"]:
            return "PROFIT"
        # Prioritas 2: Cek apakah pernah menyentuh/turun ke Cut Loss
        elif low <= row["cut_loss"]:
            return "LOSS"
        # Prioritas 3: Jika tidak kena dua-duanya
        else:
            return "HOLD"

    df["Status Evaluasi"] = df.apply(evaluasi_prioritas, axis=1)

    # ==============================================================================
    # 5. TAMPILKAN HASIL EVALUASI (JUPYTER / COLAB DISPLAY)
    # ==============================================================================
    print("\n=== HASIL EVALUASI TRADING PLAN ===")
    display(df[["code", "buy", "cut_loss", "lock", "Harga High", "Harga Low", "Status Evaluasi"]])

Membaca 10 saham... Mengambil data High & Low dari Yahoo Finance...


[*********************100%***********************]  10 of 10 completed


=== HASIL EVALUASI TRADING PLAN ===


,code,buy,cut_loss,lock,Harga High,Harga Low,Status Evaluasi
0,JARR,3710,3599,3896,3710.0,3710.0,HOLD
1,GOLF,176,171,185,189.0,171.0,PROFIT
2,BMTR,119,115,125,120.0,114.0,LOSS
3,TUGU,1435,1392,1507,1445.0,1395.0,HOLD
4,SEMA,132,128,139,163.0,130.0,PROFIT
5,WIRG,80,78,84,85.0,74.0,PROFIT
6,FOLK,252,244,265,266.0,228.0,PROFIT
7,KOCI,129,125,135,139.0,120.0,PROFIT
8,ASLI,420,407,441,442.0,400.0,PROFIT
9,AGII,3180,3085,3339,3230.0,3020.0,LOSS
